# 情感序列 × 价格收益：滞后相关与格兰杰因果

本 Notebook 调用 `src.quant_signal`，读取 `data/processed.jsonl`（由 `scripts/seed_demo_data.py` 或实时管道生成），
聚合「币安」的小时级情感序列，并与 BTCUSDT 1h K 线对齐。

默认使用**模拟 K 线**（离线可复现）：模拟收益率在 t 期受 t-1 期情感驱动，用于验证估计器能否恢复已知的领先滞后结构，
不代表真实市场结论。传入 `live=True` 会改为从 Binance 公共 API 拉取真实 K 线。

> 格兰杰检验使用 NumPy 直接实现 OLS F 检验，避免额外 SciPy/statsmodels 依赖；
> 原假设 H0：情感不格兰杰导致价格收益。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.quant_signal import run_analysis

result = run_analysis(
    ROOT / 'data' / 'processed.jsonl',
    entity='币安',
    symbol='BTCUSDT',
    freq='1h',
    max_lag=12,
    live=False,
    output=ROOT / 'reports' / 'sentiment_price_correlation.png',
    csv_output=ROOT / 'data' / 'sentiment_price.csv',
)
result

## 滞后相关结果

`correlation[k] = corr(sentiment_t, log_return_{t+k})`。绿色柱表示 p < 0.05。
若峰值出现在 lag=1，说明情感变化领先下一小时收益，而非同期同步波动。

In [ ]:
import pandas as pd

pd.DataFrame(result['lag_table']).sort_values('correlation', key=abs, ascending=False).head(12)

## 格兰杰因果检验

对每个 lag 比较「仅用收益历史」与「收益历史 + 情感历史」的残差平方和：

$$F = \frac{(RSS_r - RSS_u)/p}{RSS_u/(n-2p-1)}$$

若 p < 0.05，则拒绝「情感不导致收益」的原假设。注意：格兰杰因果是**预测意义上的领先关系**，不是物理因果。

In [ ]:
import json
print(json.dumps(result['granger'], ensure_ascii=False, indent=2))

## 可视化

四联图：情感 vs 收盘价、负面占比 vs 情感、滞后相关、情感(t) vs 收益(t+1) 散点与拟合线。

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports' / 'sentiment_price_correlation.png')))

## 结论与局限

1. 模拟数据中 lag=1 的相关性最高且格兰杰 p 值显著，说明方法能识别「情感领先价格」的结构。
2. 真实数据下需控制市场整体行情、宏观事件、交易量等混杂因素，并做平稳性/协整检验。
3. 情感来自 ABSA + Tri-Agent 的最终分数；若 LLM 路由率变化，序列可能出现结构性断点，应做分段稳健性检验。
4. 任何交易信号都必须经过样本外回测与交易成本评估，不能仅凭相关性下单。